In [15]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path

In [16]:
ARTIFACTS_DIR = Path("artifacts")
RESULTS_DIR = Path("results")
FIG_DIR = Path("figures")

In [4]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [5]:
class CXRDataset(Dataset):
    def __init__(self, df, image_col="image_path", label_col="pneumonia", transform=None):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.label_col = label_col
        self.transform = transform or preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row[self.image_col]
        y = int(row[self.label_col])

        img = Image.open(path).convert("RGB")   # important
        x = self.transform(img)
        return x, y, idx

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [10]:
from torchvision import models
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Remove classification head → output 2048-d embeddings
model.fc = nn.Identity()

model.eval().to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /voc/work/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 316MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [17]:
#extract embeddings and save them

emb_path = ARTIFACTS_DIR/"embeddings.npy"
lab_path = ARTIFACTS_DIR/"labels.npy"
meta_path = ARTIFACTS_DIR/"meta.parquet"

if emb_path.exists() and lab_path.exists() and meta_path.exists():
    X = np.load(emb_path)
    y = np.load(lab_path)
    meta = pd.read_parquet(meta_path)
    print("Loaded cached embeddings:", X.shape)
else:
    ds = CXRDataset(df, image_col=IMAGE_COL, label_col=LABEL_COL, transform=preprocess)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

    N = len(ds)
    X = np.zeros((N, 2048), dtype=np.float32)
    y = np.zeros((N,), dtype=np.int64)

    with torch.no_grad():
        for xb, yb, idx in tqdm(dl, total=len(dl)):
            xb = xb.to(device, non_blocking=True)
            feats = net(xb).detach().cpu().numpy().astype(np.float32)
            idx_np = idx.numpy()
            X[idx_np] = feats
            y[idx_np] = yb.numpy()

    meta = df.reset_index(drop=True).copy()
    meta.to_parquet(meta_path, index=False)
    np.save(emb_path, X)
    np.save(lab_path, y)
    print("Saved:", emb_path, lab_path, meta_path)

NameError: name 'df' is not defined